In [0]:
import requests

datasets = {
    "2020-10-06": "opsd_2020_june.csv",
}

base_url = "https://data.open-power-system-data.org/time_series/"
volume_folder = "/Volumes/workspace/default/my_volume/"

for release_date, file_name in datasets.items():
    full_url = f"{base_url}{release_date}/time_series_60min_singleindex.csv"
    save_path = f"{volume_folder}{file_name}"
    
    print(f"Downloading data from release: {release_date}...")
    
    response = requests.get(full_url)
    
    if response.status_code == 200:
        with open(save_path, "wb") as f:
            f.write(response.content)
        print(f"Successfully saved to {save_path}")
    else:
        print(f"Failed to download {release_date}. Status code: {response.status_code}")

In [0]:
df_opsd = spark.read.csv(f"{volume_folder}opsd_2020_june.csv", header=True, inferSchema=True)
display(df_opsd.limit(3))

In [0]:
df_opsd = spark.read.format("csv") \
    .option("header", "true") \
    .option("inferSchema", "true") \
    .load(f"{volume_folder}opsd_2020_june.csv")


df_opsd.write.mode("overwrite").saveAsTable("workspace.bronze.opsd_power_data")

print("OSPD data loaded in the Bronze layer.")

In [0]:
cities_config = {
    "BE": {"Brussels": (50.85, 4.35), "Antwerp": (51.21, 4.40), "Ghent": (51.05, 3.73)},
    "CH": {"Zurich": (47.37, 8.54), "Geneva": (46.20, 6.14), "Basel": (47.56, 7.59)},
    "DE": {"Berlin": (52.52, 13.41), "Hamburg": (53.55, 9.99), "Munich": (48.14, 11.58)},
    "ES": {"Madrid": (40.41, -3.70), "Barcelona": (41.39, 2.17), "Valencia": (39.47, -0.38)},
    "FR": {"Paris": (48.85, 2.35), "Marseille": (43.30, 5.37), "Lyon": (45.76, 4.84)},
    "GB": {"London": (51.51, -0.13), "Birmingham": (52.48, -1.89), "Glasgow": (55.86, -4.26)},
    "IT": {"Rome": (41.90, 12.50), "Milan": (45.46, 9.19), "Naples": (40.85, 14.27)},
    "NO": {"Oslo": (59.91, 10.75), "Bergen": (60.39, 5.32), "Trondheim": (63.43, 10.40)},
}

In [0]:
import requests
import time
import pandas as pd

all_weather_dfs = []
max_retries = 3


for country_code, cities in cities_config.items():
    for city_name, coords in cities.items():
        lat, lon = coords
        
        print(f"Fetching data for {city_name}, {country_code}...")
        
        params = {
            "latitude": lat,
            "longitude": lon,
            "start_date": "2015-01-01",
            "end_date": "2020-10-09",
            "hourly": "temperature_2m,windspeed_10m,cloudcover",
            "timezone": "UTC"
        }

        success = False
        attempt = 0
        
        while not success and attempt < max_retries:
                    response = requests.get("https://archive-api.open-meteo.com/v1/archive", params=params)
                    
                    if response.status_code == 200:
                        data = response.json()["hourly"]
                        pdf = pd.DataFrame(data)
                        pdf["city"], pdf["country"] = city_name, country_code
                        all_weather_dfs.append(pdf)
                        success = True
                        time.sleep(2) 
                        
                    elif response.status_code == 429:
                        attempt += 1
                        wait_time = attempt * 10 
                        print(f"Rate limited on {city_name}. Retrying in {wait_time}s (Attempt {attempt}/{max_retries})")
                        time.sleep(wait_time)
                        
                    else:
                        print(f"Failed {city_name} with status {response.status_code}")
                        break 

spark_weather_df = spark.createDataFrame(pd.concat(all_weather_dfs, ignore_index=True))
spark_weather_df.write.mode("overwrite").saveAsTable("bronze.weather_multi_count")
            

In [0]:
from pyspark.sql import functions as F

df_weather_silver = spark.table("workspace.bronze.weather_multi_count") \
    .withColumn("time", F.to_timestamp("time", "yyyy-MM-dd'T'HH:mm")) \
    .dropna(subset=["time"])

df_weather_silver.write.mode("overwrite").saveAsTable("silver.weather_multi_count_silver")

In [0]:
df_opsd_silver = spark.table("workspace.bronze.opsd_power_data") \
    .select(
        F.to_timestamp("utc_timestamp").alias("time"),
        
        F.col("FR_load_actual_entsoe_transparency").alias("FR_load"),
        F.col("FR_solar_generation_actual").alias("FR_solar"),
        F.col("FR_wind_onshore_generation_actual").alias("FR_wind"),
        
        F.col("DE_load_actual_entsoe_transparency").alias("DE_load"),
        F.col("DE_solar_generation_actual").alias("DE_solar"),
        F.col("DE_wind_generation_actual").alias("DE_wind"), 
        F.col("DE_LU_price_day_ahead").alias("DE_price"),   
        
        F.col("IT_load_actual_entsoe_transparency").alias("IT_load"),
        F.col("IT_solar_generation_actual").alias("IT_solar"),
        F.col("IT_wind_onshore_generation_actual").alias("IT_wind"),
        
        F.col("ES_load_actual_entsoe_transparency").alias("ES_load"),
        F.col("ES_solar_generation_actual").alias("ES_solar"),
        F.col("ES_wind_onshore_generation_actual").alias("ES_wind")
    )


In [0]:
display(df_opsd_silver.limit(10))

In [0]:
display(df_opsd_silver.summary("count"))

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

df_long = df_opsd_silver.select(
    "time",
    F.expr("""stack(4, 
        'FR', FR_load, FR_solar, FR_wind,
        'DE', DE_load, DE_solar, DE_wind,
        'IT', IT_load, IT_solar, IT_wind,
        'ES', ES_load, ES_solar, ES_wind
    ) as (country, load, solar, wind)""")
)

window_spec = Window.partitionBy("country").orderBy("time")

df_silver_cleaned = df_long \
    .withColumn("load", F.last("load", True).over(window_spec)) \
    .withColumn("solar", F.last("solar", True).over(window_spec)) \
    .withColumn("wind", F.last("wind", True).over(window_spec))

df_silver_final = df_silver_cleaned.dropna(subset=["load"])

In [0]:
display(df_silver_final)

In [0]:
df_gold = df_silver_final.join(
    df_weather_silver, 
    on=["time", "country"], 
    how="inner"
)

df_gold.write.mode("overwrite").saveAsTable("gold.power_risk_features")

In [0]:
display(spark.table("workspace.gold.power_risk_features"))

In [0]:
from pyspark.sql import functions as F

thresholds = df_gold.groupBy("country").agg(
    F.percentile_approx("load", 0.95).alias("high_load_threshold")
)

df_final_gold = df_gold.join(thresholds, on="country") \
    .withColumn("is_high_risk", F.when(F.col("load") >= F.col("high_load_threshold"), 1).otherwise(0)) \
    .withColumn("hour", F.hour("time")) \
    .withColumn("month", F.month("time")) \
    .withColumn("day_of_week", F.dayofweek("time")) \
    .fillna(0, subset=["solar", "wind", "load"])

df_final_gold.write.mode("overwrite").saveAsTable("workspace.gold.power_risk_features")

Databricks visualization. Run in Databricks to view.

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

thresholds = df_gold.groupBy("country").agg(
    F.percentile_approx("load", 0.95).alias("high_load_threshold")
)

df_with_risk = df_gold.join(thresholds, on="country")

df_with_risk = df_with_risk.withColumn(
    "is_high_risk", 
    F.when(F.col("load") >= F.col("high_load_threshold"), 1).otherwise(0)
)

In [0]:
from pyspark.ml.feature import VectorAssembler
from pyspark.ml import Pipeline

assembler = VectorAssembler(inputCols=feature_cols, outputCol="features", handleInvalid="skip")

pipeline = Pipeline(stages=[assembler])
pipeline_model = pipeline.fit(df_for_ml)

pipeline_model.write().overwrite().save("/Volumes/workspace/default/my_volume/weather_pipeline")

print("Pipeline successfully saved!")